# Lead Scoring Case Study — X Education
## Logistic Regression (Beginner-Friendly Version)

**Goal:** X Education gets a lot of website leads, but only about 30% convert into paying customers.
We will build a simple **Logistic Regression** model that looks at a lead's details and predicts the
**probability that the lead will convert (`Converted` = 1)**. We then turn that probability into a
**Lead Score from 0 to 100**, so the sales team can call the highest-scoring (hottest) leads first.

**Steps we will follow:**
1. Import libraries and load the data
2. Explore the data (shape, data types, missing values)
3. Clean the data (handle missing values and useless columns)
4. Simple EDA (a few basic plots)
5. Convert categorical (text) columns into numbers (dummy variables)
6. Split into train and test sets
7. Scale the numeric columns
8. Build the Logistic Regression model
9. Evaluate the model (accuracy, confusion matrix, ROC-AUC)
10. Create the final Lead Score
11. Conclusion

## Step 1: Import libraries and load the data

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import MinMaxScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import confusion_matrix, accuracy_score, classification_report, roc_auc_score, roc_curve

%matplotlib inline


In [ ]:
# Load the CSV file into a DataFrame
df = pd.read_csv('Leads.csv')

# Look at the first 5 rows
df.head()


## Step 2: Explore the data

In [ ]:
# How many rows and columns do we have?
df.shape


In [ ]:
# Data types and non-null counts of each column
df.info()


In [ ]:
# The target column we want to predict is 'Converted' (1 = converted, 0 = not converted)
df['Converted'].value_counts()


In [ ]:
# Percentage of leads that converted
print("Conversion rate:", round(df['Converted'].mean() * 100, 2), "%")


In [ ]:
# How many missing (null) values does each column have?
df.isnull().sum().sort_values(ascending=False).head(20)


**Note:** A lot of the categorical columns also have a value called `'Select'`. This just means
the customer didn't pick anything from the dropdown — it is basically the same as a missing value, so we
will treat it as `NaN` too.

In [ ]:
# Replace 'Select' with NaN (missing value) everywhere in the DataFrame
df = df.replace('Select', np.nan)


## Step 3: Clean the data

In [ ]:
# Prospect ID and Lead Number are just identifiers — they don't help predict conversion, so drop them
df = df.drop(['Prospect ID', 'Lead Number'], axis=1)


In [ ]:
# Check missing value % again
missing_pct = round(df.isnull().sum() / len(df) * 100, 2)
missing_pct[missing_pct > 0].sort_values(ascending=False)


Some columns are missing more than **40%** of their values (`Lead Quality`, the `Asymmetrique`
columns, `Tags`, `Lead Profile`, `How did you hear about X Education`, `Country`). There isn't enough
reliable data in these columns to use them confidently, so we drop them.

We also noticed some Yes/No columns (like `Magazine`, `Newspaper`, `Do Not Call`) have almost the **same
answer in 99%+ of rows**. A column where almost everyone answers the same way can't help the model tell
leads apart, so we drop those too.

In [ ]:
# Drop columns with too many missing values
cols_high_missing = ['Lead Quality', 'Asymmetrique Activity Index', 'Asymmetrique Profile Index',
                      'Asymmetrique Activity Score', 'Asymmetrique Profile Score',
                      'Tags', 'Lead Profile', 'How did you hear about X Education', 'Country']

# Drop columns where almost every row has the same value (no useful information)
cols_low_variance = ['Do Not Call', 'Search', 'Magazine', 'Newspaper Article', 'X Education Forums',
                      'Newspaper', 'Digital Advertisement', 'Through Recommendations',
                      'Receive More Updates About Our Courses', 'Update me on Supply Chain Content',
                      'Get updates on DM Content', 'I agree to pay the amount through cheque',
                      'What matters most to you in choosing a course']

df = df.drop(columns=cols_high_missing + cols_low_variance)
df.shape


In [ ]:
# For the remaining columns with a few missing values, fill them in:
# - text/category columns -> fill with the most common value (mode)
# - number columns -> fill with the median

df['Specialization'] = df['Specialization'].fillna('Others')
df['What is your current occupation'] = df['What is your current occupation'].fillna('Unemployed')
df['City'] = df['City'].fillna(df['City'].mode()[0])
df['Lead Source'] = df['Lead Source'].fillna(df['Lead Source'].mode()[0])
df['Last Activity'] = df['Last Activity'].fillna(df['Last Activity'].mode()[0])
df['TotalVisits'] = df['TotalVisits'].fillna(df['TotalVisits'].median())
df['Page Views Per Visit'] = df['Page Views Per Visit'].fillna(df['Page Views Per Visit'].median())

# Confirm there are no missing values left
df.isnull().sum().sum()


In [ ]:
# Small fix: 'google' and 'Google' are the same source, just written differently
df['Lead Source'] = df['Lead Source'].replace({'google': 'Google'})


## Step 4: Simple EDA (Exploratory Data Analysis)

In [ ]:
# Does more time spent on the website mean more conversions?
plt.figure(figsize=(6,4))
sns.boxplot(x='Converted', y='Total Time Spent on Website', data=df)
plt.title('Time spent on website vs Conversion')
plt.show()


**Observation:** Leads that convert (`Converted = 1`) tend to spend a lot more time on the website than leads that don't convert.

In [ ]:
# Which Lead Origin brings in the most conversions?
plt.figure(figsize=(6,4))
sns.countplot(x='Lead Origin', hue='Converted', data=df)
plt.xticks(rotation=45)
plt.title('Lead Origin vs Conversion')
plt.show()


In [ ]:
# Which occupation converts best?
plt.figure(figsize=(6,4))
sns.countplot(x='What is your current occupation', hue='Converted', data=df)
plt.xticks(rotation=45)
plt.title('Occupation vs Conversion')
plt.show()


**Observation:** Working professionals convert at a noticeably higher rate than students or unemployed leads.

## Step 5: Convert categorical columns into numbers (dummy variables)

Logistic Regression needs numbers, not text. For Yes/No columns we map them to 1/0. For other text columns (like `Lead Origin`, `Lead Source`) we create **dummy variables** — one new 0/1 column per category — using `pd.get_dummies()`.

In [ ]:
# Yes/No columns -> 1/0
df['Do Not Email'] = df['Do Not Email'].map({'Yes': 1, 'No': 0})
df['A free copy of Mastering The Interview'] = df['A free copy of Mastering The Interview'].map({'Yes': 1, 'No': 0})


In [ ]:
# Find all remaining text (categorical) columns
cat_cols = df.select_dtypes(include='object').columns.tolist()
cat_cols


In [ ]:
# Create dummy variables for these columns
# drop_first=True avoids creating redundant columns
dummy_df = pd.get_dummies(df[cat_cols], drop_first=True, dtype=int)

# Combine with the rest of the numeric columns
df_final = pd.concat([df.drop(columns=cat_cols), dummy_df], axis=1)
df_final.shape


## Step 6: Split into train and test sets

In [ ]:
X = df_final.drop('Converted', axis=1)   # all the input features
y = df_final['Converted']                # the target we want to predict

X_train, X_test, y_train, y_test = train_test_split(
    X, y, train_size=0.7, test_size=0.3, random_state=100
)

print("Training set size:", X_train.shape)
print("Testing set size :", X_test.shape)


## Step 7: Scale the numeric columns

`TotalVisits`, `Total Time Spent on Website`, and `Page Views Per Visit` are on very different scales (e.g. seconds vs. counts). We rescale them to a 0–1 range with `MinMaxScaler` so no single column dominates the model just because of its scale.

In [ ]:
scaler = MinMaxScaler()

num_cols = ['TotalVisits', 'Total Time Spent on Website', 'Page Views Per Visit']

X_train[num_cols] = scaler.fit_transform(X_train[num_cols])
X_test[num_cols] = scaler.transform(X_test[num_cols])


## Step 8: Build the Logistic Regression model

In [ ]:
model = LogisticRegression(max_iter=1000)
model.fit(X_train, y_train)

print("Model trained successfully!")


## Step 9: Evaluate the model

In [ ]:
# Predict on the training data first
y_train_pred = model.predict(X_train)

print("Training Accuracy:", accuracy_score(y_train, y_train_pred))


In [ ]:
# Predict on the test data (data the model has never seen)
y_test_pred = model.predict(X_test)

print("Testing Accuracy:", accuracy_score(y_test, y_test_pred))


In [ ]:
# Confusion matrix: rows = actual, columns = predicted
cm = confusion_matrix(y_test, y_test_pred)

plt.figure(figsize=(5,4))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=['Predicted: Not Converted', 'Predicted: Converted'],
            yticklabels=['Actual: Not Converted', 'Actual: Converted'])
plt.title('Confusion Matrix (Test Data)')
plt.show()


In [ ]:
# Precision, recall, F1-score for both classes
print(classification_report(y_test, y_test_pred))


In [ ]:
# ROC-AUC score: how well the model separates converted vs not converted (1.0 = perfect, 0.5 = random guess)
y_test_prob = model.predict_proba(X_test)[:, 1]   # probability of Converted = 1
auc_score = roc_auc_score(y_test, y_test_prob)
print("ROC-AUC Score:", round(auc_score, 3))


In [ ]:
# Plot the ROC curve
fpr, tpr, thresholds = roc_curve(y_test, y_test_prob)

plt.figure(figsize=(6,6))
plt.plot(fpr, tpr, label=f'ROC Curve (AUC = {auc_score:.2f})')
plt.plot([0, 1], [0, 1], 'k--', label='Random guess')
plt.xlabel('False Positive Rate')
plt.ylabel('True Positive Rate')
plt.title('ROC Curve')
plt.legend()
plt.show()


## Step 10: Create the final Lead Score (0–100)

We take the model's predicted probability of conversion (a number between 0 and 1) and multiply it by 100 to get a Lead Score. A higher score means a hotter lead.

In [ ]:
# Get predicted probability for every lead in the dataset
X_scaled_full = X.copy()
X_scaled_full[num_cols] = scaler.transform(X_scaled_full[num_cols])

all_probabilities = model.predict_proba(X_scaled_full)[:, 1]

lead_score_df = pd.DataFrame({
    'Converted': y,
    'Conversion_Probability': all_probabilities,
    'Lead_Score': (all_probabilities * 100).round(0).astype(int)
})

lead_score_df.head(10)


In [ ]:
# Check: leads that actually converted should have a higher average score
lead_score_df.groupby('Converted')['Lead_Score'].mean()


## Step 11: Conclusion

- We cleaned the leads data by removing columns with too many missing values or with almost no useful
  information, and filled in the remaining missing values.
- We converted all text columns into numeric dummy variables so Logistic Regression could use them.
- We trained a Logistic Regression model that predicts the probability a lead will convert.
- The model reached about **80% accuracy** on data it had never seen before (the test set), with a strong
  **ROC-AUC score**, meaning it is good at telling converted and non-converted leads apart.
- We used the predicted probability to build a **Lead Score (0–100)** for every lead — leads that actually
  converted have a much higher average score than leads that didn't, which confirms the score is useful.

**Business recommendation:** The sales team should prioritize calling leads with the **highest Lead Scores**
first, since those are the leads most likely to convert into paying customers.